In [1]:
!pip install earthengine-api requests tqdm pandas numpy tensorflow geopandas torch --quiet

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
GITHUB_TOKEN = user_secrets.get_secret("GITHUB_TOKEN")
GOOGLE_API_KEY = user_secrets.get_secret("GOOGLE_API_KEY")
WANDB_API_KEY = user_secrets.get_secret("WANDB_API_KEY")


In [3]:
import ee, io, json, time, urllib.request, numpy as np, pandas as pd, requests, tensorflow as tf
from datetime import datetime, timedelta
import torch
import torch.nn.functional as F

# ==============================================================================
# 1. GEE AUTHENTICATION & CONFIGURATION
# ==============================================================================
SERVICE_ACCOUNT = 'hazardnet-ee-service-kaggle@hazardnet-aas48424.iam.gserviceaccount.com'
CREDENTIALS_PATH = '/kaggle/input/datasets/ashifahmedshuvo/ee-token-json/hazardnet-aas48424-48d18edabfcc.json'

try:
    ee.Initialize(ee.ServiceAccountCredentials(SERVICE_ACCOUNT, CREDENTIALS_PATH))
    print(" GEE Initialized via Service Account")
except Exception as e:
    print(f" GEE Init Failed: {e}")

BAND_NAMES = ['SAR_VV', 'SAR_VH', 'Blue', 'Red', 'NIR', 'SWIR', 
              'Temp_2m', 'Precip', 'Max_Temp', 'Min_Temp', 
              'Soil_W1', 'Soil_W3', 'Soil_T1', 'Dewpoint', 'Solar_Rad']

MODEL_PATH = '/kaggle/input/notebooks/ashifahmedshuvo/hazardnet-model-conversion/HazardNet_Deployment_Bundles/deployment_bundle/hazardnet_fp32.tflite'
STATS_PATH = '/kaggle/input/datasets/ashifahmedshuvo/hazardnet-datasets/tensors_output/normalization_stats.json'
OUTPUT_CSV = '/kaggle/working/hazardnet_advisories_latest.csv'

HORIZONS = {'7_days': 7, '15_days': 15}
HAZARD_CLASSES = ['Cold Wave', 'Drought', 'Fire', 'Flash Flood', 'Flood', 'Heat Wave', 'Severe Local Storm', 'Tropical Cyclone']

# ==============================================================================
# 2. BANGLADESH-CALIBRATED PHYSICS TRACKS (Ground Truth Alignment)
# ==============================================================================
def bd_physics_cold_wave(tmin_c):
    # BMD anchors: 16C (watch) -> 4C (extreme)
    xs = [4, 6, 8, 10, 13, 16][::-1] # Reverse for np.interp (must be increasing)
    ys = [1.0, 0.9, 0.7, 0.5, 0.3, 0.1][::-1]
    return float(np.interp(tmin_c, xs, ys, left=1.0, right=0.1))

def bd_physics_heat_wave(tmax_c):
    # BMD anchors: 36C (watch) -> 44C (extreme)
    return float(np.interp(tmax_c, [36, 38, 40, 42, 44], [0.25, 0.50, 0.70, 0.85, 1.0], left=0.0, right=1.0))

def bd_physics_flood(rain_mm):
    # FFWC/BMD heavy rain anchors
    return float(np.interp(rain_mm, [44, 88, 150, 250], [0.40, 0.65, 0.85, 1.0], left=0.0, right=1.0))

def bd_physics_cyclone(wind_kmh):
    # NIO scale 3-min sustained winds
    return float(np.interp(wind_kmh, [63, 89, 118, 166, 221], [0.25, 0.50, 0.70, 0.85, 1.0], left=0.0, right=1.0))

def bd_physics_storm(wind_kmh):
    # Kalbaishakhi gust classes
    return float(np.interp(wind_kmh, [45, 61, 91, 121, 150], [0.25, 0.40, 0.65, 0.90, 1.0], left=0.0, right=1.0))

def bd_physics_fire(temp_max_c, wind_kmh):
    # Humid-zone FWI proxy
    heat = np.clip((temp_max_c - 25.0) / 15.0, 0.0, 1.0)
    wind = np.clip((wind_kmh - 5.0) / 20.0, 0.0, 1.0)
    return float(np.clip(0.6 * heat + 0.4 * wind, 0.0, 1.0))

# ==============================================================================
# 3. GEE SPATIAL VULNERABILITY PIPELINE
# ==============================================================================
def get_hybrid_optical(region, start, end):
    s2 = ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED').filterBounds(region).filterDate(start, end).filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', 30))
    if s2.size().getInfo() > 0:
        return s2.median().select(['B2', 'B4', 'B8', 'B11'], ['Blue', 'Red', 'NIR', 'SWIR']).unmask(0)
    l8 = ee.ImageCollection('LANDSAT/LC08/C02/T1_L2').filterBounds(region).filterDate(start, end).filter(ee.Filter.lt('CLOUD_COVER', 30))
    if l8.size().getInfo() > 0:
        return l8.median().select(['SR_B2', 'SR_B4', 'SR_B5', 'SR_B6'], ['Blue', 'Red', 'NIR', 'SWIR']).unmask(0)
    return ee.Image.constant([0, 0, 0, 0]).rename(['Blue', 'Red', 'NIR', 'SWIR']).float()

def get_spatial_step(region, start, end):
    s1 = ee.ImageCollection('COPERNICUS/S1_GRD').filterBounds(region).filterDate(start, end).filter(ee.Filter.eq('instrumentMode', 'IW'))
    s1_img = s1.median().select(['VV', 'VH']).unmask(0) if s1.size().getInfo() > 0 else ee.Image.constant([0, 0]).rename(['VV', 'VH'])
    opt = get_hybrid_optical(region, start, end)
    era5 = ee.ImageCollection('ECMWF/ERA5_LAND/DAILY_AGGR').filterBounds(region).filterDate(start, end).median()
    
    # Map ERA5 to CNN bands (Kelvin / Meters / Joules)
    era5_mapped = era5.select([
        'temperature_2m', 'total_precipitation_sum', 'temperature_2m_max', 
        'temperature_2m_min', 'volumetric_soil_water_layer_1', 'volumetric_soil_water_layer_3', 
        'soil_temperature_level_1', 'dewpoint_temperature_2m', 'surface_solar_radiation_downwards_sum'
    ], BAND_NAMES[6:15]).unmask(0)
    
    return s1_img.addBands(opt).addBands(era5_mapped).float().clip(region)

def download_numpy(image, region):
    url = image.getDownloadURL({'region': region, 'scale': 10, 'format': 'NPY'})
    data = np.load(io.BytesIO(urllib.request.urlopen(url).read()), allow_pickle=True)
    img_np = np.stack([data[b] for b in BAND_NAMES], axis=0)
    if img_np.shape[1] != 64 or img_np.shape[2] != 64:
        tensor = F.interpolate(torch.from_numpy(img_np).float().unsqueeze(0), size=(64, 64), mode='bilinear', align_corners=False)
        img_np = tensor.squeeze(0).numpy()
    return img_np

# ==============================================================================
# 4. OPEN-METEO METEOROLOGICAL TRIGGERS
# ==============================================================================
def get_openmeteo_forecast(lat, lon, days):
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat, "longitude": lon, "timezone": "Asia/Dhaka", "forecast_days": min(days + 1, 16),
        "daily": "temperature_2m_mean,temperature_2m_max,temperature_2m_min,precipitation_sum,wind_speed_10m_max"
    }
    try:
        resp = requests.get(url, params=params, timeout=10).json()
        daily = resp.get('daily', {})
        return {
            'Temp_2m_K': float(np.nanmean(daily.get('temperature_2m_mean', [25]))) + 273.15, # C -> K
            'Max_Temp_K': float(np.nanmax(daily.get('temperature_2m_max', [30]))) + 273.15,
            'Min_Temp_K': float(np.nanmin(daily.get('temperature_2m_min', [15]))) + 273.15,
            'Precip_m': float(np.nansum(daily.get('precipitation_sum', [0]))) / 1000.0,     # mm -> m
            'Precip_mm': float(np.nansum(daily.get('precipitation_sum', [0]))),
            'Wind_Max_kmh': float(np.nanmax(daily.get('wind_speed_10m_max', [0]))),
            # Celsius versions for Physics Tracks
            'Min_Temp_C': float(np.nanmin(daily.get('temperature_2m_min', [15]))),
            'Max_Temp_C': float(np.nanmax(daily.get('temperature_2m_max', [30])))
        }
    except: return None

# ==============================================================================
# 5. INFERENCE & HYBRID COGNITIVE FUSION
# ==============================================================================
with open(STATS_PATH, 'r') as f: NORM_STATS = json.load(f)
interpreter = tf.lite.Interpreter(model_path=MODEL_PATH)
interpreter.allocate_tensors()
in_det, out_det = interpreter.get_input_details()[0], interpreter.get_output_details()

def run_inference(tensor):
    interpreter.set_tensor(in_det['index'], tensor)
    interpreter.invoke()
    logits = interpreter.get_tensor(out_det[0]['index'])[0]
    sev = interpreter.get_tensor(out_det[1]['index'])[0]
    probs = tf.nn.softmax(logits).numpy()
    return HAZARD_CLASSES[np.argmax(probs)], float(np.max(probs)), float(sev)

def load_fao_gaul_districts():
    gaul = ee.FeatureCollection('FAO/GAUL/2015/level2').filter(ee.Filter.eq('ADM0_NAME', 'Bangladesh'))
    feats = gaul.map(lambda f: f.set({'lon': f.geometry().centroid().coordinates().get(0), 'lat': f.geometry().centroid().coordinates().get(1)})).getInfo()['features']
    return [{"id": i+1, "name": p.get('ADM2_NAME'), "division": p.get('ADM1_NAME'), "lat": float(p['lat']), "lon": float(p['lon'])} for i, f in enumerate(sorted(feats, key=lambda x: x['properties'].get('ADM2_NAME', ''))) for p in [f['properties']]]

# ==============================================================================
# 6. MAIN EXECUTION LOOP
# ==============================================================================
DISTRICTS = load_fao_gaul_districts()
results = []

print(f"\n Starting Edge-First Advisory Pipeline for {len(DISTRICTS)} Districts...")
for dist in DISTRICTS:
    print(f"Processing {dist['name']}...")
    region = ee.Geometry.Point([dist['lon'], dist['lat']]).buffer(320).bounds().getInfo()
    
    # 1. Fetch 10 Historical Spatial Steps (Vulnerability Assessment)
    spatial_steps = []
    today = datetime.now()
    for t in range(10, 0, -1):
        end_d = today - timedelta(days=(t-1)*10)
        start_d = end_d - timedelta(days=10)
        try:
            img = get_spatial_step(region, start_d.strftime('%Y-%m-%d'), end_d.strftime('%Y-%m-%d'))
            spatial_steps.append(download_numpy(img, region))
            time.sleep(0.2) # GEE Rate Limit Protection
        except: 
            spatial_steps.append(np.zeros((15, 64, 64)))
            
    if len(spatial_steps) != 10: continue

    # 2. Process Horizons
    for h_name, days in HORIZONS.items():
        om = get_openmeteo_forecast(dist['lat'], dist['lon'], days)
        if not om: continue
        
        # Normalize Spatial Tensor (CNN evaluates past 10 steps)
        norm_tensor = np.zeros((10, 15, 64, 64), dtype=np.float32)
        for c, band in enumerate(BAND_NAMES):
            norm_tensor[:, c, :, :] = (np.stack(spatial_steps, axis=0)[:, c, :, :] - NORM_STATS[band]['mean']) / max(NORM_STATS[band]['std'], 1e-6)
        
        tflite_in = np.expand_dims(np.transpose(norm_tensor, (0, 2, 3, 1)), axis=0)
        cnn_hazard, cnn_conf, cnn_sev = run_inference(tflite_in)
        
        # 3. Physics Track Evaluation (Future Meteorological Trigger)
        phys_sev = 0.0
        if cnn_hazard == 'Cold Wave': phys_sev = bd_physics_cold_wave(om['Min_Temp_C'])
        elif cnn_hazard == 'Heat Wave': phys_sev = bd_physics_heat_wave(om['Max_Temp_C'])
        elif cnn_hazard in ['Flood', 'Flash Flood']: phys_sev = bd_physics_flood(om['Precip_mm'])
        elif cnn_hazard == 'Tropical Cyclone': phys_sev = bd_physics_cyclone(om['Wind_Max_kmh'])
        elif cnn_hazard == 'Severe Local Storm': phys_sev = bd_physics_storm(om['Wind_Max_kmh'])
        elif cnn_hazard == 'Fire': phys_sev = bd_physics_fire(om['Max_Temp_C'], om['Wind_Max_kmh'])
        elif cnn_hazard == 'Drought': phys_sev = max(0.0, 1.0 - (om['Precip_mm'] / 100.0)) # Simple deficit proxy
        
        # 4. Trust-Gated Fusion (Hybrid Cognitive Architecture)
        # If CNN confidence is low (OOD shift), trust the Physics Track
        final_severity = (cnn_conf * cnn_sev) + ((1.0 - cnn_conf) * phys_sev)
        advisory_tier = "SEVERE" if final_severity >= 0.70 else "WARNING" if final_severity >= 0.50 else "WATCH" if final_severity >= 0.30 else "NORMAL"
        
        results.append({
            'district': dist['name'], 'division': dist['division'], 'horizon': h_name,
            'hazard': cnn_hazard, 'confidence': round(cnn_conf, 3),
            'cnn_severity': round(cnn_sev, 3), 'physics_severity': round(phys_sev, 3),
            'final_severity': round(final_severity, 3), 'advisory_tier': advisory_tier,
            'target_date': (today + timedelta(days=days)).strftime('%Y-%m-%d'),
            'om_max_temp_c': round(om['Max_Temp_C'], 1), 'om_min_temp_c': round(om['Min_Temp_C'], 1),
            'om_precip_mm': round(om['Precip_mm'], 1), 'om_wind_kmh': round(om['Wind_Max_kmh'], 1)
        })

pd.DataFrame(results).to_csv(OUTPUT_CSV, index=False)
print(f"\n Advisory Pipeline Complete. {len(results)} alerts generated.")
print(pd.DataFrame(results).sort_values('final_severity', ascending=False).head(10).to_string())

*** Earth Engine *** Share your feedback by taking our Annual Developer Satisfaction Survey: https://google.qualtrics.com/jfe/form/SV_9oS0DRcPvElRMNw?source=python


 GEE Initialized via Service Account


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.



🚀 Starting Edge-First Advisory Pipeline for 64 Districts...
Processing Bagerhat...


2026-09-19 16:41:13.875718: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Processing Bandarban...
Processing Barguna...
Processing Barisal...
Processing Bhola...
Processing Bogra...
Processing Brahamanbaria...
Processing Chandpur...
Processing Chittagong...
Processing Chuadanga...
Processing Comilla...
Processing Cox's Bazar...
Processing Dhaka...
Processing Dinajpur...
Processing Faridpur...
Processing Feni...
Processing Gaibandha...
Processing Gazipur...
Processing Gopalganj...
Processing Habiganj...
Processing Jamalpur...
Processing Jhalokati...
Processing Jhenaidah...
Processing Joypurhat...
Processing Khagrachhari...
Processing Khulna...
Processing Kishoreganj...
Processing Kurigram...
Processing Kushtia...
Processing Lakshmipur...
Processing Lalmonirhat...
Processing Madaripur...
Processing Magura...
Processing Manikganj...
Processing Maulvibazar...
Processing Meherpur...
Processing Munshiganj...
Processing Mymensingh...
Processing Naogaon...
Processing Narail...
Processing Narayanganj...
Processing Narsingdi...
Processing Natore...
Processing Nawabgan